# 📖 Module 01: RAG Fundamentals

## GenAI L2 Exam Preparation

**Topics Covered:**
- What is RAG and why it exists
- RAG vs Fine-tuning — when to use which
- End-to-end pipeline architecture
- Key terminology
- Common exam scenarios

**Source Material:** Class 29 (RAG Introduction), Class 31 (RAG Pipeline End-to-End)

---

## 1. What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances LLM responses by combining:
- **Retrieval**: Finding relevant documents/information from a knowledge base
- **Generation**: Using an LLM to generate answers based on the retrieved context

### The Problem RAG Solves

LLMs have inherent limitations:

| Problem | Description | How RAG Helps |
|---------|-------------|---------------|
| **Hallucination** | LLMs can generate plausible but incorrect information | Grounds responses in actual documents |
| **Knowledge Cutoff** | Training data has a fixed end date | Retrieves up-to-date information |
| **No Private Data** | LLMs don't know your company's internal docs | Connects to your private knowledge base |
| **No Source Attribution** | Hard to verify where info came from | Provides source documents with answers |

### 🎯 Exam Tip
> RAG = **"Give the LLM a cheat sheet before answering"**  
> Instead of relying on memorized knowledge (training data), we retrieve relevant context first.

## 2. RAG Pipeline Architecture

The RAG pipeline has two main phases:

### Phase 1: Indexing (Offline — done once)
```
Documents → Load → Chunk → Embed → Store in Vector DB
```

### Phase 2: Querying (Online — per user query)
```
User Query → Embed Query → Retrieve Similar Chunks → Augment Prompt → LLM → Response
```

### Complete Pipeline Diagram

```
┌─────────────────────── INDEXING PHASE ───────────────────────┐
│                                                               │
│  📄 Documents    →   📝 Chunks    →   🔢 Embeddings   →  🗄️ Vector DB  │
│  (PDF, DOCX,        (Split text      (Convert to           (Store for    │
│   HTML, CSV)         into pieces)     numerical vectors)    fast search)  │
│                                                               │
└───────────────────────────────────────────────────────────────┘

┌─────────────────────── QUERYING PHASE ───────────────────────┐
│                                                               │
│  ❓ User Query  →  🔢 Query Embedding  →  🔍 Retrieve Top-K  │
│                                              relevant chunks   │
│                                                    ↓           │
│  📤 Response  ←  🤖 LLM Generation  ←  📋 Augmented Prompt   │
│  (with sources)   (Generate answer)    (Query + Context)      │
│                                                               │
└───────────────────────────────────────────────────────────────┘
```

### 🎯 Exam Tip
> Know the **6 stages**: Load → Chunk → Embed → Store → Retrieve → Generate  
> Be able to identify which stage has a problem given a scenario.

## 3. ⭐ RAG vs Fine-Tuning (HIGH PRIORITY — Exam Favorite!)

This is one of the **most commonly asked exam questions**.

| Aspect | RAG | Fine-Tuning |
|--------|-----|-------------|
| **Purpose** | Add external knowledge at inference time | Modify model behavior/style permanently |
| **Data freshness** | ✅ Always up-to-date (real-time retrieval) | ❌ Frozen at training time |
| **Cost** | Low (no GPU training needed) | High (requires GPU training) |
| **Setup time** | Fast (hours) | Slow (days/weeks) |
| **Hallucination** | Reduced (grounded in documents) | Can still hallucinate |
| **Domain knowledge** | Via document retrieval | Baked into model weights |
| **Model size** | Uses existing model as-is | Creates new model weights |
| **Source attribution** | ✅ Can cite sources | ❌ No source tracking |
| **Best for** | Q&A over docs, knowledge bases, chatbots | Style transfer, format control, specialized tasks |

### Decision Framework
```
Need up-to-date information?        → RAG
Need to access private documents?   → RAG
Need source citations?              → RAG
Need to change output style/format? → Fine-tuning
Need specialized domain behavior?   → Fine-tuning (or RAG + few-shot prompting)
Need both?                          → RAG + Fine-tuned model (best of both)
```

### 🎯 Exam Tip
> If the question mentions "company internal documents", "latest data", or "source attribution" → **RAG**  
> If it mentions "change model behavior", "specific output format", or "domain-specific language" → **Fine-tuning**

## 4. Key Terminology (Exam Definitions)

| Term | Definition |
|------|------------|
| **Indexing** | The process of converting documents into searchable vector representations |
| **Chunking** | Splitting documents into smaller pieces for better retrieval precision |
| **Embedding** | Converting text into numerical vectors that capture semantic meaning |
| **Vector Store / Vector DB** | Database optimized for storing and searching vector embeddings |
| **Retriever** | Component that finds the most relevant chunks for a given query |
| **Augmentation** | Adding retrieved context to the user's query before sending to LLM |
| **Grounding** | Ensuring LLM responses are based on retrieved evidence, not imagination |
| **Top-K** | Number of most similar documents retrieved (e.g., k=5 means top 5 results) |
| **Semantic Search** | Finding documents based on meaning similarity (not keyword matching) |
| **Context Window** | Maximum number of tokens an LLM can process in a single request |
| **Chain** | A sequence of LLM operations (e.g., retrieve → prompt → generate) |

## 5. Hands-On: Minimal RAG Pipeline

Let's build the simplest possible RAG pipeline to see all stages in action.

In [4]:
# Setup: Load environment variables
from dotenv import load_dotenv
load_dotenv()

import os
print("✅ Environment loaded")

✅ Environment loaded


In [5]:
os.environ.get('GROQ_API_KEY')

'gsk_***REDACTED***'

In [6]:
# Stage 1: LOAD — Create sample documents
from langchain_core.documents import Document

documents = [
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval with LLM generation.", metadata={"source": "doc1"}),
    Document(page_content="Vector databases store embeddings and enable fast similarity search. Examples include FAISS, ChromaDB, and Pinecone.", metadata={"source": "doc2"}),
    Document(page_content="Fine-tuning modifies model weights on custom data. It's different from RAG which retrieves information at inference time.", metadata={"source": "doc3"}),
    Document(page_content="Chunking splits documents into smaller pieces. Common strategies include fixed-size, recursive, and semantic chunking.", metadata={"source": "doc4"}),
    Document(page_content="Embeddings convert text into numerical vectors. Similar texts produce vectors that are close in vector space.", metadata={"source": "doc5"}),
]

print(f"✅ Loaded {len(documents)} documents")
print(f"📄 Sample: {documents[0].page_content[:80]}...")

✅ Loaded 5 documents
📄 Sample: RAG stands for Retrieval-Augmented Generation. It combines retrieval with LLM ge...


In [7]:
# Stage 2: EMBED + STORE — Create embeddings and store in vector DB
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Create embeddings (free, runs locally)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in FAISS (in-memory vector DB)
vectorstore = FAISS.from_documents(documents, embeddings)

print(f"✅ Created vector store with {len(documents)} vectors")
print(f"📐 Embedding dimension: {len(embeddings.embed_query('test'))}")

c:\Users\anilc\Desktop\GenAI_bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\anilc\AppData\Local\Temp\ipykernel_2288\979670654.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ Created vector store with 5 vectors
📐 Embedding dimension: 384


In [8]:
# Stage 3: RETRIEVE — Find relevant documents for a query
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

query = "What is the difference between RAG and fine-tuning?"
retrieved_docs = retriever.invoke(query)

print(f"🔍 Query: {query}")
print(f"📄 Retrieved {len(retrieved_docs)} documents:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"  [{i}] {doc.page_content}")
    print(f"      Source: {doc.metadata['source']}\n")

🔍 Query: What is the difference between RAG and fine-tuning?
📄 Retrieved 2 documents:

  [1] Fine-tuning modifies model weights on custom data. It's different from RAG which retrieves information at inference time.
      Source: doc3

  [2] RAG stands for Retrieval-Augmented Generation. It combines retrieval with LLM generation.
      Source: doc1



In [9]:
# Stage 4: GENERATE — Use LLM with retrieved context
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize LLM (Groq is free and fast)
llm = ChatGroq(model="openai/gpt-oss-20b")

# Create RAG prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question based ONLY on the provided context. If the context doesn't contain the answer, say 'I don't have enough information.'"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# Build the chain
chain = prompt | llm | StrOutputParser()

# Format context from retrieved docs
context = "\n".join([doc.page_content for doc in retrieved_docs])

# Generate answer
try:
    answer = chain.invoke({"context": context, "question": query})
    print(f"❓ Question: {query}\n")
    print(f"🤖 Answer: {answer}")
except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Make sure GROQ_API_KEY is set in your .env file")

⚠️ Error: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
Make sure GROQ_API_KEY is set in your .env file


## 6. RAG Pipeline Components — Deep Dive

### Component Responsibilities

| Component | Input | Output | Key Decision |
|-----------|-------|--------|-------------|
| **Document Loader** | File path | List of `Document` objects | Which loader to use |
| **Text Splitter** | Documents | Smaller chunks | chunk_size, overlap |
| **Embedding Model** | Text string | Vector (list of floats) | Which model, dimensions |
| **Vector Store** | Vectors + metadata | Indexed database | Which DB, distance metric |
| **Retriever** | Query string | Top-K relevant documents | search_type, k value |
| **LLM** | Prompt + context | Generated text | Model selection, temperature |
| **Chain** | All of the above | End-to-end pipeline | Chain type (stuff, etc.) |

### 🎯 Exam Tip
> For scenario questions: identify which **component** is responsible for the issue.  
> - Bad retrieval results? → Check **Retriever** settings or **Chunking** strategy  
> - Hallucinations? → Check if **context is being used** in the prompt  
> - Slow performance? → Check **Vector DB** choice or **embedding model** size  
> - Out of token limit? → Check **chunk_size** or **Top-K** value

## 7. LangChain — The Framework for RAG

LangChain is the most popular framework for building RAG pipelines. Key concepts:

### Core Abstractions

| Abstraction | What it does | Example |
|-------------|-------------|----------|
| `Document` | Holds text + metadata | `Document(page_content="...", metadata={})` |
| `DocumentLoader` | Loads files into Documents | `PyPDFLoader("file.pdf")` |
| `TextSplitter` | Splits Documents into chunks | `RecursiveCharacterTextSplitter()` |
| `Embeddings` | Converts text → vectors | `HuggingFaceEmbeddings()` |
| `VectorStore` | Stores & searches vectors | `FAISS.from_documents()` |
| `Retriever` | Finds relevant docs for a query | `vectorstore.as_retriever()` |
| `ChatModel` | LLM interface | `ChatGroq()`, `ChatOpenAI()` |
| `PromptTemplate` | Formats prompts with variables | `ChatPromptTemplate.from_messages()` |
| `Chain` | Connects components together | `RetrievalQA.from_chain_type()` |
| `OutputParser` | Parses LLM output | `StrOutputParser()` |

### LCEL (LangChain Expression Language)
```python
# Modern way to build chains using the pipe (|) operator
chain = prompt | llm | output_parser
result = chain.invoke({"question": "What is RAG?"})
```

### 🎯 Exam Tip
> Know the difference between:
> - `langchain` — Main framework
> - `langchain_core` — Core abstractions (prompts, output parsers)
> - `langchain_community` — Community integrations (loaders, vector stores)
> - Provider packages — `langchain_groq`, `langchain_openai`, etc.

## 🧠 Self-Assessment Quiz

Test yourself on these questions before moving to the next module.

---

**Q1.** What are the two main phases of a RAG pipeline?

<details>
<summary>Click for Answer</summary>

1. **Indexing Phase** (offline): Load → Chunk → Embed → Store  
2. **Querying Phase** (online): Embed Query → Retrieve → Augment → Generate
</details>

---

**Q2.** A company wants to build a chatbot that answers questions about their 10,000-page internal policy manual that gets updated weekly. Should they use RAG or Fine-tuning?

<details>
<summary>Click for Answer</summary>

**RAG** — because:  
- The data is private (internal policy manual)  
- The data changes frequently (updated weekly)  
- They need answers grounded in specific documents  
- RAG can handle updates without retraining the model
</details>

---

**Q3.** What does "augmented" mean in Retrieval-Augmented Generation?

<details>
<summary>Click for Answer</summary>

"Augmented" means the LLM prompt is **enhanced/augmented** with retrieved context before generation. The user's question is combined with relevant documents to give the LLM the necessary information to answer accurately.
</details>

---

**Q4.** An LLM keeps generating outdated information about a rapidly changing topic. Which RAG component would directly solve this?

<details>
<summary>Click for Answer</summary>

The **Retriever** + regularly updated **Vector Store**. By keeping the knowledge base updated with fresh documents and retrieving from it at query time, the LLM always gets current information.
</details>

---

**Q5.** What is the role of embeddings in a RAG pipeline?

<details>
<summary>Click for Answer</summary>

Embeddings convert text into **numerical vectors** (lists of numbers) that capture semantic meaning. This allows us to:  
- Store text in a vector database  
- Compute similarity between query and documents  
- Retrieve the most semantically relevant chunks
</details>

---

## ✅ Module 1 Complete!

**Key Takeaways:**
1. RAG = Retrieval + Augmentation + Generation
2. Two phases: Indexing (offline) and Querying (online)
3. RAG vs Fine-tuning: RAG for knowledge, FT for behavior
4. 6 pipeline stages: Load → Chunk → Embed → Store → Retrieve → Generate
5. LangChain provides abstractions for each stage

**Next:** [Module 02 — Data Parsing & Document Loaders](./02_Data_Parsing_and_Loaders.ipynb)